# Zero-shot T5-small summarisation on CNN/DailyMail (200 test articles)
T5-small vs a Lead-3 baseline, scored with ROUGE (F1).

## 1. Install packages

In [ ]:
%pip install -q transformers datasets rouge-score sentencepiece matplotlib pandas torch nltk tqdm

## 2. Imports

In [ ]:
import os, time, random, json, platform, shutil, textwrap
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import nltk
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from rouge_score import rouge_scorer

nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

## 3. Config (seed, device, parameters)

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

MODEL_NAME = "t5-small"

MAX_INPUT_TOKENS  = 512
MAX_OUTPUT_TOKENS = 150
MIN_OUTPUT_TOKENS = 40
NUM_BEAMS         = 4
LENGTH_PENALTY    = 2.0

SUBSET_SIZE = 200
HUMAN_EVAL_SIZE = 20     # exported scaffold size for human ratings

## 4. Load model and tokenizer

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print(f"Loaded {MODEL_NAME}")

## 5. Load CNN/DailyMail (non-anonymized 3.0.0) and choose the subset

In [ ]:
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")
test_data = dataset["test"]
print(test_data)

test_subset = test_data.shuffle(seed=SEED).select(range(min(SUBSET_SIZE, len(test_data))))
print(f"Using {len(test_subset)} test examples.")

## 6. Generator (zero-shot, with the "summarize:" prefix)

In [ ]:
def generate_summary(article_text: str) -> str:
    input_text = "summarize: " + article_text
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_TOKENS,
        truncation=True
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
            max_length=MAX_OUTPUT_TOKENS,
            min_length=MIN_OUTPUT_TOKENS,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.1
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True)

## 7. ROUGE scorer (F1)

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

## 8. Main loop: generate and score with T5-small

In [ ]:
rows = []
for i, ex in enumerate(tqdm(test_subset, desc="Summarising")):
    article   = ex["article"]
    reference = ex["highlights"]

    t0 = time.time()
    generated = generate_summary(article)
    dt = time.time() - t0

    scores = scorer.score(reference, generated)
    rows.append({
        "id": i,
        "article": article,
        "reference_summary": reference,
        "generated_summary": generated,
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
        "gen_time_sec": dt,
        "device": str(DEVICE)
    })

df = pd.DataFrame(rows)
print(df[["rouge1","rouge2","rougeL","gen_time_sec"]].describe().round(3))

## 9. Save T5-small results

In [ ]:
OUT_CSV = "summarization_results_output.csv"
df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")

## 10. Lead-3 baseline (first 3 sentences of each article, same 200 articles)

In [ ]:
base_rows = []
for i, ex in enumerate(tqdm(test_subset, desc="Lead-3")):
    lead3 = " ".join(sent_tokenize(ex["article"])[:3])
    s = scorer.score(ex["highlights"], lead3)
    base_rows.append({
        "id": i,
        "rouge1": s["rouge1"].fmeasure,
        "rouge2": s["rouge2"].fmeasure,
        "rougeL": s["rougeL"].fmeasure,
    })

baseline_df = pd.DataFrame(base_rows)
baseline_df.to_csv("lead3_baseline_results.csv", index=False)
print("Saved: lead3_baseline_results.csv")
print(baseline_df[["rouge1","rouge2","rougeL"]].mean().round(3))

## 11. T5-small vs Lead-3 with bootstrap 95% confidence intervals

In [ ]:
metrics = ["rouge1", "rouge2", "rougeL"]
rng = np.random.default_rng(SEED)
N_BOOT = 2000

def boot_ci(values, iters=N_BOOT, alpha=0.05):
    vals = np.asarray(values)
    boots = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(iters)])
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return vals.mean(), lo, hi

# df and baseline_df cover the same articles in the same order (id 0..n-1)
assert (df["id"].values == baseline_df["id"].values).all()

table = []
for m in metrics:
    t_mean, t_lo, t_hi = boot_ci(df[m].values)
    b_mean, b_lo, b_hi = boot_ci(baseline_df[m].values)
    diff = df[m].values - baseline_df[m].values
    d_mean, d_lo, d_hi = boot_ci(diff)
    table.append({
        "metric": m,
        "T5-small": f"{t_mean:.3f} ({t_lo:.3f}, {t_hi:.3f})",
        "Lead-3": f"{b_mean:.3f} ({b_lo:.3f}, {b_hi:.3f})",
        "T5 minus Lead-3": f"{d_mean:+.3f} ({d_lo:+.3f}, {d_hi:+.3f})",
    })

comparison = pd.DataFrame(table)
print("Mean ROUGE F1 with bootstrap 95% CI. If the CI of the difference excludes 0, the gap is unlikely to be chance.")
print(comparison.to_string(index=False))
comparison.to_csv("t5_vs_lead3_comparison.csv", index=False)

## 12. Summary stats and plots

In [ ]:
stats = df[metrics + ["gen_time_sec"]].agg(["mean", "std", "min", "max"]).round(3)
print(stats)
stats.to_csv("results_summary_stats.csv")

avg = pd.DataFrame({
    "T5-small": df[metrics].mean(),
    "Lead-3": baseline_df[metrics].mean(),
})
avg.plot(kind="bar", figsize=(7, 4))
plt.title("Average ROUGE (F1): T5-small zero-shot vs Lead-3 (200 test articles)")
plt.ylabel("F1"); plt.ylim(0, 1.0); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("fig_avg_rouge.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6, 4))
df[metrics].boxplot()
plt.title("T5-small ROUGE distribution per sample (F1)")
plt.ylabel("F1"); plt.ylim(0, 1.0); plt.tight_layout()
plt.savefig("fig_rouge_boxplots.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(df["gen_time_sec"].dropna(), bins=20)
plt.title("Generation time per sample (seconds)")
plt.xlabel("Seconds"); plt.ylabel("Count"); plt.tight_layout()
plt.savefig("fig_time_hist.png", dpi=200, bbox_inches="tight")
plt.show()

df["gen_len_words"] = df["generated_summary"].str.split().str.len()
plt.figure(figsize=(6, 4))
plt.scatter(df["gen_len_words"], df["rougeL"], alpha=0.6)
plt.title("Generated summary length vs ROUGE-L (F1)")
plt.xlabel("Generated summary length (words)"); plt.ylabel("ROUGE-L (F1)")
plt.ylim(0, 1.0); plt.tight_layout()
plt.savefig("fig_len_vs_rougeL.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Best and worst examples by ROUGE-L

In [ ]:
def show_examples(frame, k=3, sort_col="rougeL", ascending=False, wrap=100):
    ex = frame.sort_values(sort_col, ascending=ascending).head(k)
    for _, row in ex.iterrows():
        print("=" * 80)
        print(f"{sort_col.upper()}: {row[sort_col]:.3f}")
        print("- Reference summary:")
        print(textwrap.fill(row["reference_summary"], wrap))
        print("- Generated summary:")
        print(textwrap.fill(row["generated_summary"], wrap))
        print()

print("TOP examples by ROUGE-L:\n")
show_examples(df, k=3, ascending=False)
print("BOTTOM examples by ROUGE-L:\n")
show_examples(df, k=3, ascending=True)

In [ ]:
def md_escape(s):
    return str(s).replace("|", "\\|").replace("`", "\\`")

k = min(5, len(df))
top5 = df.sort_values("rougeL", ascending=False).head(k)
bot5 = df.sort_values("rougeL", ascending=True).head(k)

lines = ["# Summarization Examples Report\n", "## Top examples by ROUGE-L\n"]
for _, r in top5.iterrows():
    lines += [f"**ROUGE-L:** {r['rougeL']:.3f}",
              f"- **Reference:** {md_escape(r['reference_summary'])}",
              f"- **Generated:** {md_escape(r['generated_summary'])}", ""]
lines.append("\n## Bottom examples by ROUGE-L\n")
for _, r in bot5.iterrows():
    lines += [f"**ROUGE-L:** {r['rougeL']:.3f}",
              f"- **Reference:** {md_escape(r['reference_summary'])}",
              f"- **Generated:** {md_escape(r['generated_summary'])}", ""]

with open("examples_report.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("Saved: examples_report.md")

## 14. Crude hallucination flag
Capitalised words in the summary that never appear in the article. This is only a rough heuristic, not a measured hallucination rate.

In [ ]:
def cap_tokens(s):
    return {t.strip(",.?!:;") for t in s.split() if t[:1].isupper() and len(t) > 2}

def hallucination_flag(article, summary):
    extra = cap_tokens(summary) - cap_tokens(article)
    stop_caps = {"The","A","An","He","She","They","It","I","In","On","At","And","But","For","To"}
    extra = {e for e in extra if e not in stop_caps}
    return int(len(extra) > 0)

df["halluc_flag_crude"] = [hallucination_flag(a, g) for a, g in zip(df["article"], df["generated_summary"])]
print(f"Summaries with at least one unseen capitalised word: {df['halluc_flag_crude'].sum()} of {len(df)} ({df['halluc_flag_crude'].mean():.1%})")

## 15. Human evaluation scaffold (ratings must be filled in by hand)

In [ ]:
he_size = min(HUMAN_EVAL_SIZE, len(df))
he_subset = df.sample(n=he_size, random_state=SEED).copy()

HUMAN_EVAL_CSV = "human_eval_scaffold.csv"
he_subset[["id", "generated_summary", "reference_summary"]].assign(**{
    "Fluency(1-5)": "",
    "Coherence(1-5)": "",
    "Faithfulness(1-5)": "",
    "Hallucination(0/1)": "",
    "Notes": ""
}).to_csv(HUMAN_EVAL_CSV, index=False)
print(f"Saved: {HUMAN_EVAL_CSV}  (fill in the rating columns manually, then re-run the next cell)")

In [ ]:
HUMAN_RESULTS = "human_eval_scaffold.csv"
human_cols = ["Fluency(1-5)", "Coherence(1-5)", "Faithfulness(1-5)"]

if os.path.exists(HUMAN_RESULTS):
    hr = pd.read_csv(HUMAN_RESULTS)
    if hr[human_cols].isna().all().all():
        print("No human ratings provided - skipping")
    else:
        merged = pd.merge(df, hr, on="id", how="inner")
        print(f"Merged human ratings: {len(merged)} rows")
        print(merged[human_cols].mean().round(2))
        halluc = merged["Hallucination(0/1)"].dropna()
        if len(halluc):
            print(f"Hallucination rate (human): {halluc.mean():.1%}")
else:
    print("No human ratings provided - skipping")

## 16. Run metadata

In [ ]:
import transformers
meta = {
    "device": str(DEVICE),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "subset_size": len(test_subset),
    "model": MODEL_NAME,
    "seed": SEED,
    "params": {
        "max_input": MAX_INPUT_TOKENS,
        "max_output": MAX_OUTPUT_TOKENS,
        "min_output": MIN_OUTPUT_TOKENS,
        "num_beams": NUM_BEAMS,
        "length_penalty": LENGTH_PENALTY
    }
}
with open("run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

## 17. Results pack

In [ ]:
os.makedirs("results_pack", exist_ok=True)
for f in ["summarization_results_output.csv", "lead3_baseline_results.csv", "t5_vs_lead3_comparison.csv",
          "results_summary_stats.csv", "fig_avg_rouge.png", "fig_rouge_boxplots.png",
          "fig_time_hist.png", "fig_len_vs_rougeL.png", "run_metadata.json",
          "human_eval_scaffold.csv", "examples_report.md"]:
    if os.path.exists(f):
        shutil.copy(f, os.path.join("results_pack", f))
print("Packed files in ./results_pack")

## 18. RESULTS SUMMARY

In [ ]:
def fmt(values):
    m, lo, hi = boot_ci(values)
    return f"{m:.3f} (95% CI {lo:.3f}-{hi:.3f})"

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"Number of articles: {len(df)}")
print(f"Device: {DEVICE}")
print(f"Mean generation time per summary: {df['gen_time_sec'].mean():.2f} s")
print()
print("T5-small (zero-shot):")
for m in metrics:
    print(f"  {m.upper():8s} {fmt(df[m].values)}")
print("Lead-3 baseline:")
for m in metrics:
    print(f"  {m.upper():8s} {fmt(baseline_df[m].values)}")
print("=" * 60)